# 第9回 強化学習 — 正解を教えずに、うまくなってもらう

**このノートブックは第9回だけで完結します。** プログラミングの経験は要りません。

やること:
1. スロットで「ためすか、いま最善を取るか」を比べる
2. マス目を歩かせて、地図なしで最短の道を見つけさせる
3. **報酬の書きかたを変えて、わざと壊してみる**(今日の山場)
4. 自分で報酬を設計して、意図しない行動を探す

所要時間:実習 約25分 / 課題 約40分

> 外部のモデルもデータも使いません。すべてこのノートブックの中で完結します。

## 0. 準備

上から順にセルを実行してください(セルを選んで **Shift + Enter**)。

In [ ]:
# 準備(このセルを最初に1回だけ実行)
!pip install -q japanize-matplotlib
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt, japanize_matplotlib
plt.rcParams["font.size"] = 12
print("準備できました")

## 1. 探索と活用 — ためすか、いま最善を取るか

5台のスロットがあります。当たる確率は台ごとに違いますが、**AI は事前に知りません**。
引きながら覚えるしかない。1000回引けます。

`ε`(イプシロン)は「でたらめに選ぶ確率」です。
- `ε = 0` … 常に、これまでの成績がいちばん良い台を選ぶ
- `ε = 1` … 常に、でたらめに選ぶ

In [ ]:
# ===== ① 探索と活用:5台のスロット =====
真の当たり率 = [0.20, 0.35, 0.55, 0.40, 0.30]   # AI にはこれを教えません
最良の台 = int(np.argmax(真の当たり率))

def 引き続ける(ε, 回数=1000, seed=0):
    """ε の確率ででたらめに選び、それ以外は「これまでの成績が最良」の台を選ぶ"""
    乱数 = np.random.default_rng(seed)
    引いた回数 = np.zeros(5)
    これまでの平均 = np.zeros(5)
    総報酬 = 0
    最良を選んだ回数 = 0
    for t in range(回数):
        if 乱数.random() < ε:
            台 = int(乱数.integers(5))               # ためす(探索)
        else:
            台 = int(np.argmax(これまでの平均))       # いま最善(活用)
        当たり = 1 if 乱数.random() < 真の当たり率[台] else 0
        引いた回数[台] += 1
        これまでの平均[台] += (当たり - これまでの平均[台]) / 引いた回数[台]
        総報酬 += 当たり
        最良を選んだ回数 += (台 == 最良の台)
    return 総報酬, 最良を選んだ回数 / 回数 * 100, 引いた回数

print(f"5台の当たり率: {真の当たり率}   いちばん良いのは {最良の台+1}番台\n")
print(f"{'方針':<22}{'総報酬':>8}{'最良台を選んだ率':>16}")
print("-" * 48)
結果 = {}
for 名, ε in [("ε=0    常に最善", 0.0), ("ε=0.05  5%ためす", 0.05),
              ("ε=0.1  10%ためす", 0.1), ("ε=0.3  30%ためす", 0.3),
              ("ε=1.0  でたらめ", 1.0)]:
    R = [引き続ける(ε, seed=s) for s in range(50)]      # 50回くり返して平均
    報酬 = np.mean([r[0] for r in R]); 率 = np.mean([r[1] for r in R])
    結果[名] = 報酬
    print(f"{名:<22}{報酬:>8.0f}{率:>15.1f}%")
print(f"\n上限(最良台だけを引く): {真の当たり率[最良の台]*1000:.0f}")
print(f"下限(最悪台だけを引く): {min(真の当たり率)*1000:.0f}")

> **`ε = 0` が最悪になった理由**を、次のセルで確かめます。

In [ ]:
# ε=0 は、どの台に居着くのか
from collections import Counter
居着き先 = Counter()
for s in range(50):
    _, _, 回数 = 引き続ける(0.0, seed=s)
    居着き先[int(np.argmax(回数)) + 1] += 1
print("ε=0 が最も多く引いた台(50回の内訳):", dict(sorted(居着き先.items())))
print("→ 最初にたまたま当たった台に居着き、他を一度も試しません。")

plt.figure(figsize=(8, 3.8))
名前 = list(結果.keys()); 値 = list(結果.values())
色 = ["#E08A2B" if v < 400 else "#3A46A8" for v in 値]
plt.bar(range(len(値)), 値, color=色)
plt.axhline(真の当たり率[最良の台]*1000, ls="--", c="#02A96A", lw=1.5)
plt.text(0.05, 真の当たり率[最良の台]*1000 + 12, "上限(最良台だけ)", color="#02A96A", fontsize=10)
plt.axhline(min(真の当たり率)*1000, ls="--", c="crimson", lw=1.5)
plt.text(0.05, min(真の当たり率)*1000 + 12, "下限(最悪台だけ)", color="crimson", fontsize=10)
plt.xticks(range(len(値)), [n.split()[0] for n in 名前])
plt.ylabel("1000回の総報酬"); plt.title("ためす割合(ε)を変えると")
plt.grid(alpha=.3, axis="y"); plt.tight_layout(); plt.show()

## 2. マス目を歩かせる — Q学習

5×5 のマス目。左上がスタート、右下がゴール。最短で 8歩です。
**AI はマス目の形も、ゴールの場所も知りません。**上下左右に動けることだけを知っています。

覚えるのは「このマスでこの向きに進んだら、最終的にいくらもらえそうか」という表(5×5×4 = 100個の数)だけです。

In [ ]:
# ===== ② マス目を歩かせる:Q学習 =====
N = 5
スタート, ゴール = (0, 0), (4, 4)
動き = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}   # 上下左右

def 進む(位置, 行動):
    r, c = 位置; dr, dc = 動き[行動]
    return (min(max(r + dr, 0), N-1), min(max(c + dc, 0), N-1))   # 壁では留まる

def Q学習(報酬の設計, 回数=3000, 上限=100, ε=0.1, 学習率=0.5, 割引率=0.95, seed=0):
    乱数 = np.random.default_rng(seed)
    Q = np.zeros((N, N, 4))          # 「このマスでこの向きに進む」見込み表
    履歴 = []
    for 回 in range(回数):
        位置 = スタート; 合計 = 0
        for 歩 in range(上限):
            行動 = int(乱数.integers(4)) if 乱数.random() < ε else int(np.argmax(Q[位置]))
            次 = 進む(位置, 行動)
            着いた = (次 == ゴール)
            報酬 = 報酬の設計(位置, 次, 着いた)
            # ↓ Q学習の更新式。これ1行だけです
            Q[位置][行動] += 学習率 * (報酬 + 割引率 * (0 if 着いた else Q[次].max()) - Q[位置][行動])
            合計 += 報酬; 位置 = 次
            if 着いた: break
        履歴.append((歩 + 1, 合計, 着いた))
    return Q, 履歴

def 正しい報酬(位置, 次, 着いた):
    return 10.0 if 着いた else -0.1      # ゴールで+10、1歩ごとに-0.1

Q, 履歴 = Q学習(正しい報酬)
歩数 = np.array([h[0] for h in 履歴])
print(f"1回目        : {歩数[0]} 歩")
for a, b in [(1,10),(10,50),(50,100),(100,300),(1000,3000)]:
    print(f"{a:>5}〜{b:<5}回目: {歩数[a:b].mean():.1f} 歩")
print(f"\n最短は 8 歩。マス目の形もゴールの場所も、一切教えていません。")

In [ ]:
# 学んだ方針と、各マスの価値を見る
矢 = ["↑", "↓", "←", "→"]
print("学んだ方針(各マスで選ぶ向き / ◎ がゴール)")
for r in range(N):
    print("   " + "  ".join("◎" if (r,c)==ゴール else 矢[int(np.argmax(Q[(r,c)]))] for c in range(N)))

print("\n各マスの価値(そこから最終的にいくらもらえる見込みか)")
for r in range(N):
    print("   " + " ".join(f"{Q[(r,c)].max():5.1f}" for c in range(N)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(歩数[:200], lw=1, color="#3A46A8")
axes[0].axhline(8, ls="--", c="crimson"); axes[0].text(120, 10, "最短 8 歩", color="crimson")
axes[0].set_xlabel("何回目"); axes[0].set_ylabel("ゴールまでの歩数")
axes[0].set_title("学習の進みかた(最初の200回)"); axes[0].grid(alpha=.3)

価値 = np.array([[Q[(r,c)].max() for c in range(N)] for r in range(N)])
im = axes[1].imshow(価値, cmap="YlGnBu")
for r in range(N):
    for c in range(N):
        axes[1].text(c, r, f"{価値[r,c]:.1f}", ha="center", va="center", fontsize=10)
axes[1].set_title("各マスの価値(濃いほど高い)"); axes[1].set_xticks([]); axes[1].set_yticks([])
plt.colorbar(im, ax=axes[1], shrink=.8)
plt.tight_layout(); plt.show()

print("誰も地図を渡していません。「ゴールで +10」と書いただけで、")
print("その価値がひとりでに手前へ広がっていきます。")

## 3. 報酬設計をまちがえる — 今日の山場

マス目もゴールも学習方法も、まったく同じにします。
**変えるのは「報酬の書きかた」だけ**です。

- **A** 正しい設計
- **B** 「ゴールの隣にいると +1」を、良かれと思って足す
- **C** 「1歩ごとに −0.1」を、うっかり書き忘れる

In [ ]:
# ===== ③ 報酬設計をまちがえる(今日の山場)=====
def 距離(位置): return abs(位置[0]-ゴール[0]) + abs(位置[1]-ゴール[1])

設計 = [
    ("A 正しい:ゴール +10 / 1歩ごとに −0.1",
     lambda s, s2, e: 10.0 if e else -0.1),
    ("B ゴールの隣にいると +1(近さをほめる)",
     lambda s, s2, e: (10.0 if e else 0.0) + (1.0 if 距離(s2) <= 1 and not e else 0.0)),
    ("C 1歩ごとの罰を書き忘れた",
     lambda s, s2, e: 10.0 if e else 0.0),
]

print(f"{'報酬の設計':<38}{'平均歩数':>9}{'報酬':>8}{'到達率':>8}")
print("-" * 66)
学習済み = {}
for 名, f in 設計:
    Q2, H = Q学習(f)
    学習済み[名] = Q2
    後 = H[-300:]
    print(f"{名:<38}{np.mean([h[0] for h in 後]):>9.1f}"
          f"{np.mean([h[1] for h in 後]):>8.1f}"
          f"{np.mean([h[2] for h in 後])*100:>7.0f}%")

print("\n【重要】報酬の数字だけを見ると、B が A の5倍うまくやったように見えます。")
print("        実際の動きを見てみましょう。")

In [ ]:
# 学習後、実際にどう動くのか(数字ではなく振る舞いを見る)
for 名, Q2 in 学習済み.items():
    位置 = スタート; 道 = [位置]
    for t in range(20):
        位置 = 進む(位置, int(np.argmax(Q2[位置])))
        道.append(位置)
        if 位置 == ゴール: break
    着いた = "ゴール到達" if 道[-1] == ゴール else "20歩たっても着かない"
    print(f"{名}")
    print(f"   {' → '.join(str(p) for p in 道[:12])}{' …' if len(道)>12 else ''}")
    print(f"   → {着いた}\n")

print("B は (3,4) ── ゴールの隣 ── に居座って +1 を稼ぎ続けています。")
print("C は (0,0) から1歩も動きません。急ぐ理由がないからです。")
print()
print("どちらも AI は悪くありません。書いたとおりに最適化しただけです。")
print("報酬という物差しの内側にいる限り、この失敗は見えません。")

> **今日いちばん持ち帰ってほしいこと**
>
> B は報酬を A の **5倍** もらいました。数字だけ見れば大成功です。
> でも、やっていたのは「ゴールの手前でうろうろすること」でした。
>
> 報酬という物差しの内側にいる限り、この失敗は見えません。
> **数字の外に出て、実際の振る舞いを見る**しかありません。
>
> これは AI に限りません。テストの点、売上、KPI、いいねの数 ──
> 測って評価した瞬間、人も組織もその数字を上げるほうへ動きます(**グッドハートの法則**)。

## 4. 自分で試す

**壊すのが目的の実験**です。変な動きを見つけたら、それが成果です。

In [ ]:
# ===== 自分で試す(課題用)=====
# ↓ ここを書きかえて、自分の報酬設計を試してください。
#    s = いまの位置、s2 = 動いたあとの位置、e = ゴールに着いたか

def 私の報酬(s, s2, e):
    return 10.0 if e else -0.1          # ← 書きかえる
    # 試してみる例:
    #   1歩ごとに +0.1 にしたら?     → return 10.0 if e else 0.1
    #   近づくたびに +0.5 にしたら?  → return (10.0 if e else 0.0) + (0.5 if 距離(s2)<距離(s) else 0.0)
    #   壁にぶつかったら −1 にしたら? → return (10.0 if e else -0.1) - (1.0 if s == s2 else 0.0)

Q3, H3 = Q学習(私の報酬)
後 = H3[-300:]
print(f"平均歩数 {np.mean([h[0] for h in 後]):.1f}   "
      f"報酬 {np.mean([h[1] for h in 後]):.1f}   "
      f"到達率 {np.mean([h[2] for h in 後])*100:.0f}%")

位置 = スタート; 道 = [位置]
for t in range(20):
    位置 = 進む(位置, int(np.argmax(Q3[位置]))); 道.append(位置)
    if 位置 == ゴール: break
print("実際の動き:", " → ".join(str(p) for p in 道[:14]))
print("\n※ 報酬の数字だけでなく、必ず「実際の動き」を確認してください。")

In [ ]:
# ===== おまけ:穴(大きな罰)を置くとどうなるか =====
穴 = [(1, 1), (2, 3), (3, 1)]

def 穴つき報酬(s, s2, e):
    if e: return 10.0
    if s2 in 穴: return -5.0
    return -0.1

Q4, _ = Q学習(穴つき報酬)
矢 = ["↑", "↓", "←", "→"]
print("穴(×)を避ける方針が学習されます:")
for r in range(N):
    行 = []
    for c in range(N):
        if (r, c) == ゴール: 行.append("◎")
        elif (r, c) in 穴:   行.append("×")
        else:                行.append(矢[int(np.argmax(Q4[(r, c)]))])
    print("   " + "  ".join(行))
print("\n誰も「穴を避けよ」とは書いていません。")
print("穴の周辺の価値が下がり、自然に迂回する道が選ばれただけです。")

## 5. 提出課題(manaba)

**締切:次回講義の前日 23:59**

### 課題1:手で計算する(電卓可)

**(a) スロット**
- D台 … 4回引いて4回当たり
- E台 … 30回引いて18回当たり
- F台 … 10回引いて5回当たり

それぞれの当たり率を求め、`ε=0` ならどれを選ぶか、**その判断の問題点**を書きなさい。

**(b) Q学習の更新を2回連続で計算しなさい**
初めの見込み 0、報酬 0、次のマスの最高の見込み 10.0、学習率 0.5、割引率 0.9。

> 式:新しい見込み = いまの見込み + 学習率 ×(報酬 + 割引率 × 次の最高の見込み − いまの見込み)

### 課題2:ノートブックを動かして報告する
- (a) ε を変えたときの総報酬のグラフ
- (b) 学習曲線と、学んだ方針・価値の図
- (c) **自分で報酬設計を1つ考えて試し、何が起きたかを書く**

> (c) は、うまくいったかどうかでは採点しません。
> **「報酬の数字」と「実際の動き」の両方を見て、そのズレを説明できているか**を見ます。

### 課題3:次の主張を批判する(300字以上)

> 「配達員の評価を AI で最適化します。報酬は『1時間あたりの配達件数』。
> 多く配れば報酬が増えるので、AI は自然に効率の良い配り方を見つけるはずです。
> 目標は明快で、公平な指標です。」

考える材料:
- 「件数」を最大化する行動には、どんなものがありますか(望ましくないものも含めて)
- 報酬に**書かれていないもの**は、AI にとってどう扱われますか(今日の C を思い出して)
- この指標は、担当エリアが違っても公平ですか
- 指標が評価に使われると、人の行動はどう変わりますか

**加点**:「では、どう設計し直すか」まで書けると評価が上がります。
ただし「完璧な報酬設計がある」と書くのは誤りです ── 直しても、また別の抜け道が見つかります。

---

採点は、正解かどうかよりも **試したことと考えたこと** を見ます。